# COVID-19 Tweet Classification Challenge
**Objective**: Develop a machine learning model to assess if a Twitter post is about COVID-19 or not.
**Metric**: Area Under the Curve (AUC)

This model helps gather tweet data about the epidemic without relying only on keywords like 'covid' or 'coronavirus'.

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Path Configuration

In [ ]:
def get_paths():
    train_path = 'Train.csv'
    test_path = 'Test.csv'
    
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'Train.csv' in files:
                train_path = os.path.join(root, 'Train.csv')
            if 'Test.csv' in files:
                test_path = os.path.join(root, 'Test.csv')
                
    return train_path, test_path

TRAIN_CSV, TEST_CSV = get_paths()
print(f"Train: {TRAIN_CSV}")
print(f"Test: {TEST_CSV}")

## 2. Dataset Class

In [ ]:
class TweetDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        
        encoding = self.tokenizer.encode_plus(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            return_token_type_ids=False,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt',
        )
        
        item = {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten()
        }
        
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
            
        return item

## 3. Training and Evaluation Functions

In [ ]:
def train_epoch(model, data_loader, optimizer, device):
    model.train()
    losses = []
    for d in tqdm(data_loader):
        input_ids = d["input_ids"].to(device)
        attention_mask = d["attention_mask"].to(device)
        labels = d["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        losses.append(loss.item())
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    return np.mean(losses)

def eval_model(model, data_loader, device):
    model.eval()
    preds = []
    actuals = []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device)
            attention_mask = d["attention_mask"].to(device)
            labels = d["labels"].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs.logits
            probs = torch.softmax(logits, dim=1)[:, 1]
            
            preds.extend(probs.cpu().numpy())
            actuals.extend(labels.cpu().numpy())
    return roc_auc_score(actuals, preds)

## 4. Main Execution

In [ ]:
if os.path.exists(TRAIN_CSV):
    train = pd.read_csv(TRAIN_CSV)
    # Zindi COVID Tweet dataset usually has 'text' and 'target' columns
    # We'll handle different column names just in case
    text_col = 'text' if 'text' in train.columns else train.columns[1]
    target_col = 'target' if 'target' in train.columns else train.columns[-1]
    
    MODEL_NAME = 'bert-base-uncased'
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_df, val_df = train_test_split(train, test_size=0.1, random_state=42)
    
    train_ds = TweetDataset(train_df[text_col].values, train_df[target_col].values, tokenizer)
    val_ds = TweetDataset(val_df[text_col].values, val_df[target_col].values, tokenizer)
    
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=16, shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)
    
    for epoch in range(3):
        loss = train_epoch(model, train_loader, optimizer, device)
        auc = eval_model(model, val_loader, device)
        print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Val AUC: {auc:.4f}")
        
    # Inference
    if os.path.exists(TEST_CSV):
        test = pd.read_csv(TEST_CSV)
        test_ds = TweetDataset(test[text_col].values, tokenizer=tokenizer)
        test_loader = DataLoader(test_ds, batch_size=16, shuffle=False)
        
        model.eval()
        test_probs = []
        with torch.no_grad():
            for d in tqdm(test_loader, desc="Predicting"):
                input_ids = d["input_ids"].to(device)
                attention_mask = d["attention_mask"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                probs = torch.softmax(outputs.logits, dim=1)[:, 1]
                test_probs.extend(probs.cpu().numpy())
        
        sub = pd.DataFrame({'ID': test['ID'], 'target': test_probs})
        sub.to_csv('submission.csv', index=False)
        print("Submission saved!")
else:
    print("Train.csv not found.")